In [1]:
%%writefile test.cu
#include <cstdio>
#include <cuda_runtime.h>

#define BLOCK_SIZE 256          // размер блока для сложения


__global__ void vecAdd(float *a, float *b, int n, float *c)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < n) {
        c[idx] = a[idx] + b[idx];
    }
}

void vecAddCPU(float* A, float* B, float* C, int n) {
    for (int i = 0; i < n; i++) {
        C[i] = A[i] + B[i];
    }
    printf("\nFirst few elements of C (CPU result):\n");
    for (int i = 0; i < (i + 5 < n ? 5 : n); i++) {
        printf("%8.2f ", C[i]);
    }
    printf("\n");
}

int main(int argc, char* argv[])
{

    int n = 1024 * 1024;        // размер вектора
    int numBytes = n * sizeof(float);

    // allocate host memory
    float *a = new float[n];
    float *b = new float[n];
    float *c = new float[n];
    float *c_cpu = new float[n];

    // Инициализация векторов
    for (int i = 0; i < n; i++) {
        a[i] = 1.0f;
        b[i] = 2.0f;
    }

    // CPU вычисление
    clock_t start_cpu = clock();
    vecAddCPU(a, b, c_cpu, n);
    clock_t end_cpu = clock();
    printf("CPU Simple version: %.2f ms\n",
           (double)(end_cpu - start_cpu) / CLOCKS_PER_SEC * 1000);

    // allocate device memory
    float *adev = NULL, *bdev = NULL, *cdev = NULL;
    cudaMalloc((void**)&adev, numBytes);
    cudaMalloc((void**)&bdev, numBytes);
    cudaMalloc((void**)&cdev, numBytes);


    dim3 threads(BLOCK_SIZE);
    dim3 blocks((n + BLOCK_SIZE - 1) / BLOCK_SIZE);

    // cuda events
    cudaEvent_t start_gpu, stop_gpu;
    cudaEventCreate(&start_gpu);
    cudaEventCreate(&stop_gpu);

    // Запуск GPU
    cudaEventRecord(start_gpu, 0);
    cudaMemcpy(adev, a, numBytes, cudaMemcpyHostToDevice);
    cudaMemcpy(bdev, b, numBytes, cudaMemcpyHostToDevice);

    vecAdd<<<blocks, threads>>>(adev, bdev, n, cdev);

    cudaMemcpy(c, cdev, numBytes, cudaMemcpyDeviceToHost);
    cudaEventRecord(stop_gpu, 0);
    cudaEventSynchronize(stop_gpu);

    float gpuTime = 0.0f;
    cudaEventElapsedTime(&gpuTime, start_gpu, stop_gpu);

    printf("\nVector size: %d\n", n);
    printf("GPU time: %.2f milliseconds\n", gpuTime);

    // Вывод результата GPU
    printf("\nFirst few elements of C (GPU result):\n");
    for (int i = 0; i < (i + 5 < n ? 5 : n); i++) {
        printf("%8.2f ", c[i]);
    }
    printf("\n");

    // Верификация
    float maxDiff = 0.0f;
    for (int i = 0; i < n; i++) {
        float diff = c_cpu[i] - c[i];
        if (diff < 0) diff = -diff;
        if (diff > maxDiff) maxDiff = diff;
    }


    // cleanup
    cudaEventDestroy(start_gpu);
    cudaEventDestroy(stop_gpu);
    cudaFree(adev);
    cudaFree(bdev);
    cudaFree(cdev);
    delete[] a;
    delete[] b;
    delete[] c;
    delete[] c_cpu;

    return 0;
}

Writing test.cu


In [2]:
!nvcc test.cu -o test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./test


First few elements of C (CPU result):
    3.00     3.00     3.00     3.00     3.00 
CPU Simple version: 5.22 ms

Vector size: 1048576
GPU time: 133.12 milliseconds

First few elements of C (GPU result):
    3.00     3.00     3.00     3.00     3.00 
Max difference CPU vs GPU: 0.000000
